# 푸드코트 AI · 특성 입력 시간 민감도 실행

본 실험에서는 손님이 키오스크에 자기 특성(맛 3개, 가격, 대기, 바쁨, 예산 = 7개)을 입력하는 시간을 따로 넣지 않았습니다. 이 노트북은 그 시간을 넣고 **대표 점수표 몇 개만** 다시 학습·테스트합니다.

- 입력 시간(`input_minutes`)은 **입력값을 읽는 추천**(AI와 규칙 추천)을 받은 손님에게만 더해집니다. 미추천, 무작위, 인기 추천은 그대로입니다. 입력을 건너뛴 손님(누락)은 시간이 들지 않고 중립값이 들어갑니다.
- 바꾸는 것은 설정 몇 개뿐이고, 시드는 본 실험과 같은 반복 번호(0번)를 씁니다. 그래서 본 실험 결과와 1:1로 비교하면 **입력 시간 때문에 생긴 차이만** 보입니다.
- 설정마다 Drive에 별도 폴더가 생기고, 한 런타임에서 모든 설정을 동시에 돌립니다(코어를 나눠 씀).

In [ ]:
# 1. 소스 가져오기 · 설치 (본 실험 소스 + 특성 입력 시간 기능만 추가한 버전)
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/phj9187/foodcourt-adaptive-sim.git'
SOURCE_COMMIT = '84d7522ec0a06b18b9458d0dfc3ec53120666889'  # 642cf41 + input_minutes (기본 0이면 본 실험과 동일)
PARTIAL_REPOSITORY = 'https://github.com/phj9187/foodcourt-partial-test.git'
WORK = Path('/content/foodcourt-adaptive-sim')
TOOL = Path('/content/foodcourt-partial-test')
if not WORK.exists():
    subprocess.run(['git','clone',REPOSITORY,str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
if not TOOL.exists():
    subprocess.run(['git','clone',PARTIAL_REPOSITORY,str(TOOL)],check=True)
else:
    subprocess.run(['git','-C',str(TOOL),'pull','-q','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(WORK)],check=True)
if str(WORK/'src') not in sys.path:
    sys.path.insert(0,str(WORK/'src'))
# 본 실험과 같은 torch 빌드
_v=subprocess.run([sys.executable,'-c','import torch;print(torch.__version__)'],capture_output=True,text=True).stdout.strip()
if not _v.startswith('2.14.1+cpu'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','torch==2.14.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    _v=subprocess.run([sys.executable,'-c','import torch;print(torch.__version__)'],capture_output=True,text=True).stdout.strip()
print('시뮬레이션 소스:',subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip())
print('부분 테스트 코드:',subprocess.check_output(['git','-C',str(TOOL),'rev-parse','HEAD'],text=True).strip())
print('torch:',_v,' CPU 코어:',os.cpu_count())

In [ ]:
# 2. Google Drive 연결 · 무엇을 돌릴지 정하기 (아직 학습하지 않음)
from google.colab import drive
import re
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/foodcourt_adaptive_sim')
REFERENCE = ROOT/'extended_main_20261108_001'    # 끝난 본 실험 (읽기만 함, 같은 시드로 비교)
CONFIG = 'config/adaptive_extended.yaml'
OBJECTIVES = 'lhs_0022,lhs_0019,customer_only,profit_only'   # 대표 점수표
REPS = '0'                                         # 본 실험의 같은 반복과 1:1 비교
# 실험 ID: 설정 묶음
SETTINGS = {
    'input_t025_m25_20261109': ['input_minutes=0.25','input_missing_probability=0.25'],
    'input_t050_m25_20261109': ['input_minutes=0.5','input_missing_probability=0.25'],
}
WORKERS = os.cpu_count() or 1
share = max(1, WORKERS//len(SETTINGS))
RUNS = {}
for experiment, sets in SETTINGS.items():
    if not re.fullmatch(r'[A-Za-z0-9_-]+',experiment): raise ValueError(experiment)
    out = ROOT/experiment; out.mkdir(parents=True,exist_ok=True)
    args=['--source',str(WORK),'--config',CONFIG,'--output',str(out),'--reps',REPS,'--workers',str(share),'--objectives',OBJECTIVES]
    for s in sets: args+=['--set',s]
    if REFERENCE.exists(): args+=['--reference',str(REFERENCE)]
    else: print('본 실험 폴더가 없어 비교 없이 새 결과만 정리합니다:',REFERENCE)
    plan=subprocess.run([sys.executable,str(TOOL/'partial_test.py'),*args,'--dry-run'],cwd=WORK,capture_output=True,text=True)
    print('=====',experiment); print(plan.stdout); print(plan.stderr[-2000:]); plan.check_returncode()
    RUNS[experiment]=(out,args)
print('설정별 병렬 작업 수:',share,'/ 전체 코어',WORKERS)

In [ ]:
# 3. 모든 설정을 동시에 실행 (같은 칸을 다시 실행하면 체크포인트부터 이어감)
from datetime import datetime, timezone
import threading
from kiosk_adaptive.study import atomic_json
procs={}
def pump(name,proc,log):
    for line in proc.stdout:
        log.write(line)
        if line.startswith(('DONE','jobs=','lhs_','customer_only','profit_only','Traceback')) or 'Error' in line:
            print(f'[{name}]',line,end='')
threads=[]
for name,(out,args) in RUNS.items():
    atomic_json(out/'run_status.json',dict(status='running',updated_utc=datetime.now(timezone.utc).isoformat()))
    log=(out/'run.log').open('a',encoding='utf-8',buffering=1)
    p=subprocess.Popen([sys.executable,'-u',str(TOOL/'partial_test.py'),*args],cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    procs[name]=(p,out,log); t=threading.Thread(target=pump,args=(name,p,log),daemon=True); t.start(); threads.append(t)
try:
    for name,(p,out,log) in procs.items():
        code=p.wait()
        atomic_json(out/'run_status.json',dict(status='complete' if code==0 else 'failed',exit_code=code,updated_utc=datetime.now(timezone.utc).isoformat()))
    for t in threads: t.join()
except KeyboardInterrupt:
    for name,(p,out,log) in procs.items():
        if p.poll() is None: p.terminate()
    print('중단됨. 다시 실행하면 마지막 저장 지점부터 이어갑니다.')

In [ ]:
# 4. 결과 — 본 실험(같은 시드)과 비교한 변화, 안정 조건
for name,(out,_) in RUNS.items():
    path=out/'partial_summary.json'
    if not path.exists(): print(name,'아직 결과 없음'); continue
    report=json.loads(path.read_text(encoding='utf-8'))
    print('=====',name)
    for oid,e in report.items():
        new=e.get('new_only',{}).get('stable'); ch=e.get('change_vs_reference',{}).get('stable')
        if new: print(f"{oid:18} 입력시간 포함: 손님 {new['customer_delta_per100']:+.2f}/100  이익 {new['profit_delta_won']:+.0f}원  F {new['F_delta']:+.5f}  AI-최선규칙 {new['ai_minus_best_rule']}")
        if ch:
            m=lambda v:sum(v)/len(v)
            print(f"{'':18} 본 실험 대비 변화: 손님 {m(ch['customer_change_per100']):+.2f}  이익 {m(ch['profit_change_won']):+.0f}원  F {m(ch['F_change']):+.5f}")